# Prédiction du Risque d'Inondation — Nord Tunisie
## PFA Machine Learning — Université iTeam — 2025/2026

---

## Introduction

Ce projet vise à développer une solution de Machine Learning supervisé pour l'estimation du risque d'inondation dans le nord de la Tunisie.

Le dataset `InondatNorthTun.csv` contient différentes variables géographiques, topographiques et climatiques décrivant les zones étudiées, notamment l'altitude, la pente, les précipitations, les distances aux cours d'eau, à la mer et aux routes, ainsi que le type d'occupation du sol.

L'objectif est de concevoir un pipeline de Machine Learning permettant de :
- préparer et analyser les données ;
- développer des variables pertinentes à travers le feature engineering ;
- sélectionner les variables les plus discriminantes ;
- entraîner et optimiser plusieurs modèles de classification ;
- comparer leurs performances à l'aide de métriques adaptées ;
- analyser les erreurs de classification et les probabilités prédites ;
- préparer le modèle pour un déploiement sous forme de service de prédiction.

**Pipeline du projet :**

1. **Exploration et préparation des données** — analyse statistique et visualisation, traitement des valeurs manquantes, analyse des distributions et des classes, préparation des variables numériques et catégorielles.
2. **Feature Engineering** — création de variables liées à la proximité de l'eau, transformations logarithmiques des variables asymétriques, création de variables combinant les caractéristiques topographiques et climatiques. *Les seuils utilisés (quantiles) sont appris uniquement sur X_train afin d'éviter toute fuite de données.*
3. **Sélection des variables** — analyse statistique des variables, exploration avec `SelectKBest` (ANOVA F-score, ajustée sur X_train uniquement).
4. **Modélisation** — Logistic Regression, Random Forest, SVM, XGBoost ; optimisation des hyperparamètres par validation croisée stratifiée (`RandomizedSearchCV`, k=5).
5. **Sélection du modèle et du scaler** — basée sur le score moyen de validation croisée (CV ROC-AUC), jamais sur le jeu de test, afin de garder X_test totalement indépendant du processus de décision.
6. **Évaluation finale** — ROC-AUC, Precision, Recall, F1-score, Accuracy, Specificity, matrices de confusion, courbes ROC et analyse des seuils de décision — calculée **une seule fois** sur X_test.
7. **Déploiement** — sauvegarde du modèle et de ses artefacts, interface interactive Gradio pour tester les prédictions.

**Architecture cible.** À terme, le projet pourra évoluer vers une architecture MLOps intégrant le suivi des expériences (MLflow), la gestion des versions du modèle, une exposition via API (FastAPI), la conteneurisation (Docker) et la surveillance du service de prédiction. Une amélioration méthodologique importante à envisager, étant donné la nature géographique du dataset, est la **validation spatiale** (spatial cross-validation) : un split aléatoire classique peut donner une estimation trop optimiste de l'AUC si des points géographiquement proches se retrouvent à la fois dans le train et dans le test.


## Étape 0 — Imports et Configuration

Tous les imports sont regroupés dans une seule cellule. L'installation de `xgboost` et `gradio` est commentée — décommenter si nécessaire.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ÉTAPE 0 — IMPORTS ET CONFIGURATION
# ═══════════════════════════════════════════════════════════════
# Décommenter si nécessaire :
# !pip install xgboost gradio

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
import os, joblib

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_val_predict,
                                     GroupShuffleSplit, GroupKFold)
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from scipy.stats import uniform, randint
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, recall_score, f1_score,
                             confusion_matrix, precision_score,
                             accuracy_score, roc_curve, ConfusionMatrixDisplay)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB = True
except:
    COLAB = False

RANDOM_SEED = 42
TEST_SIZE   = 0.20
K_BEST      = 15

print('✔ Imports OK')
print(f'  COLAB = {COLAB}')


## Étape 1 — Préparation des Données

La qualité du prétraitement conditionne directement la performance des modèles. Cette étape couvre le chargement, l'exploration, la détection des valeurs aberrantes, le feature engineering et la définition du pipeline de normalisation.

### 1.1 — Chargement et Exploration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1.1 — CHARGEMENT ET EXPLORATION
# ═══════════════════════════════════════════════════════════════

CSV_PATH = '/content/drive/MyDrive/PFA_ML/DATA/InondatNorthTun.csv'
df_raw = pd.read_csv(CSV_PATH)

print('=' * 55)
print('INFORMATIONS GÉNÉRALES DU DATASET')
print('=' * 55)
print(f'Dimensions       : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes')
print(f'Colonnes         : {list(df_raw.columns)}')
print(f'\nTypes de données :\n{df_raw.dtypes}')
print(f'\nValeurs manquantes :\n{df_raw.isnull().sum()}')
print(f'\nStatistiques descriptives :')
print(df_raw.describe())
df_raw.head(10)


**Interprétation :** Cette exploration révèle la structure du dataset. Les statistiques descriptives montrent des plages de valeurs très différentes (altitude en mètres, précipitations en mm, distances en mètres), ce qui justifie l'étape de normalisation. La présence de valeurs manquantes oriente la stratégie d'imputation (médiane, robuste aux outliers).

### 1.2 — Visualisation des Distributions et Détection des Outliers

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1.2 — DISTRIBUTIONS ET DÉTECTION DES OUTLIERS
# ═══════════════════════════════════════════════════════════════
# Trois graphiques :
#   1. Histogrammes → forme de la distribution de chaque variable
#   2. Boxplots     → détection des valeurs aberrantes (règle 1.5 × IQR)
#   3. Variable cible → équilibre des classes (flood=0 vs flood=1)

continuous_vars = ['elev_m', 'slope_d', 'precipmm', 'STREAM_DIS',
                   'SEA_DISTAN', 'ROAD_DISTA', 'CITIES_SET']
plt.style.use('seaborn-v0_8-darkgrid')

# Histogrammes
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, var in enumerate(continuous_vars):
    sns.histplot(data=df_raw, x=var, kde=True, bins=30,
                 color='cornflowerblue', ax=axes[i])
    axes[i].set_title(f'Distribution : {var}')
axes[7].axis('off')
plt.suptitle('Distributions — AVANT prétraitement', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Boxplots
fig2, axes2 = plt.subplots(2, 4, figsize=(20, 8))
axes2 = axes2.flatten()
for i, var in enumerate(continuous_vars):
    sns.boxplot(data=df_raw, y=var, ax=axes2[i], color='lightcoral')
    axes2[i].set_title(f'Outliers : {var}')
axes2[7].axis('off')
plt.suptitle('Détection des Valeurs Aberrantes (Boxplots)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Variable cible
fc = df_raw['flood'].value_counts()
fig3, ax3 = plt.subplots(figsize=(5, 4))
ax3.bar(['Non inondé (0)', 'Inondé (1)'], fc.values,
        color=['steelblue', 'firebrick'], alpha=0.85)
for i, v in enumerate(fc.values):
    ax3.text(i, v + 1, str(v), ha='center', fontweight='bold')
ax3.set_title('Distribution de la variable cible : flood')
ax3.set_ylabel('Nombre de points')
plt.tight_layout(); plt.show()

desequilibre = fc[0] / fc[1]
print(f'Classe 0 (Non inondé) : {fc[0]} ({fc[0]/len(df_raw)*100:.1f}%)')
print(f'Classe 1 (Inondé)     : {fc[1]} ({fc[1]/len(df_raw)*100:.1f}%)')
print(f'Ratio déséquilibre    : {desequilibre:.2f}')
if desequilibre < 1.5:   print('✔ Classes équilibrées — accuracy fiable.')
elif desequilibre < 4:   print('⚠ Léger déséquilibre — ROC-AUC et F1 privilégiés.')
else:                    print('⚠ Fort déséquilibre — class_weight recommandé.')
print()
print('Note sur les outliers : les valeurs extrêmes (altitude élevée, grande distance à la mer)')
print('correspondent à des réalités géographiques réelles — conservées intentionnellement.')


**Interprétation :**

- **Histogrammes** : plusieurs variables présentent une asymétrie à droite (STREAM_DIS, SEA_DISTAN), typique des données de distance. La transformation logarithmique sera appliquée lors du feature engineering pour réduire cette asymétrie et améliorer la performance des modèles linéaires.

- **Boxplots** : les valeurs extrêmes observées sont des données géographiques réelles (sommets montagneux, zones intérieures éloignées). Leur présence justifie le choix de StandardScaler plutôt que MinMaxScaler, qui serait fortement biaisé par ces extrêmes.

- **Variable cible** : le ratio de déséquilibre détermine si `class_weight='balanced'` est nécessaire. Un fort déséquilibre rendrait l'accuracy trompeuse — ROC-AUC est alors la métrique de référence.


### 1.3 — Feature Engineering

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1.3 — FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════
# Justification : les nouvelles variables capturent des relations non-linéaires
# que les modèles linéaires ne peuvent pas détecter directement.
# La fonction apply_feature_engineering() est réutilisée à l'entraînement,
# dans l'interface Gradio ET pour les nouvelles prédictions (Étape 5).
#
# ⚠ ANTI DATA LEAKAGE : deux variables binaires (flat_lowland, precip_low_elev)
# dépendent de seuils (quantiles 25%/75%). Ces seuils NE DOIVENT être appris
# que sur X_train (Étape 1.5, cf. FE_THRESHOLDS) — jamais sur tout le dataset,
# et jamais recalculés par ligne à l'inférence (voir bug corrigé ci-dessous :
# l'ancienne version renvoyait toujours 0 pour un point unique, car un quantile
# calculé sur une seule ligne n'a pas de sens).

df  = df_raw.copy()
X   = df.drop(columns=['flood', 'id'])
y   = df['flood']

def apply_feature_engineering(df_in, thresholds=None):
    """Applique toutes les transformations de feature engineering.

    Parameters
    ----------
    df_in : DataFrame brute (une ou plusieurs lignes).
    thresholds : dict optionnel {'slope_q25', 'elev_q25', 'precip_q75'}.
        Si fourni (cas normal : thresholds=FE_THRESHOLDS appris sur X_train),
        ces seuils fixes sont utilisés — comportement identique pour 1 ligne
        ou pour tout un DataFrame, donc pas de fuite de données ET pas de bug
        sur les prédictions ligne-par-ligne.
        Si None, les seuils sont recalculés à partir de df_in lui-même
        (utilisé UNIQUEMENT ici, pour l'exploration sur X avant le split —
        Étape 1.4 / analyse de corrélation).
    """
    d   = df_in.copy()
    eps = 1e-6  # Protection contre la division par zéro

    # Proximité à l'eau (relation non-linéaire : doubler la distance ne divise pas le risque par 2)
    d['InvStreamDist']         = 1 / (d['STREAM_DIS'] + eps)
    d['InvSeaDist']            = 1 / (d['SEA_DISTAN'] + eps)
    d['water_proximity_score'] = 2 * d['InvStreamDist'] + d['InvSeaDist']
    d['min_water_dist']        = d[['STREAM_DIS', 'SEA_DISTAN']].min(axis=1)

    # Indices topographiques
    d['topo_wetness']      = d['elev_m'] / (d['slope_d'] + 1)   # Zone de dépression
    d['precip_elev_ratio'] = d['precipmm'] / (d['elev_m'] + 1)  # Pluie sur zone basse

    # Accessibilité urbaine
    d['urban_accessibility'] = 1 / (d['CITIES_SET'] + d['ROAD_DISTA'] + 1)

    # Variables binaires de zones à risque élevé — seuils fixes, appris sur X_train
    if thresholds is not None:
        slope_q25, elev_q25, precip_q75 = (thresholds['slope_q25'],
                                            thresholds['elev_q25'],
                                            thresholds['precip_q75'])
    else:
        slope_q25   = d['slope_d'].quantile(0.25)
        elev_q25    = d['elev_m'].quantile(0.25)
        precip_q75  = d['precipmm'].quantile(0.75)

    d['flat_lowland']    = ((d['slope_d'] < slope_q25) & (d['elev_m'] < elev_q25)).astype(int)
    d['precip_low_elev'] = ((d['precipmm'] > precip_q75) & (d['elev_m'] < elev_q25)).astype(int)

    # Transformées logarithmiques (réduisent l'asymétrie des distances)
    for col in ['STREAM_DIS', 'SEA_DISTAN', 'ROAD_DISTA', 'CITIES_SET', 'min_water_dist']:
        d[col + '_log'] = np.log(d[col] + 1)
    return d

# Appel EDA uniquement (seuils calculés sur tout X) — sert au graphe de corrélation
# de l'Étape 1.4. Les features réellement utilisées pour l'entraînement seront
# recalculées avec des seuils appris sur X_train à l'Étape 1.5 (variable FE_THRESHOLDS).
X = apply_feature_engineering(X)

n_orig = len(df_raw.columns) - 2
print(f'Features originales : {n_orig}')
print(f'Nouvelles features  : {len(X.columns) - n_orig}')
print(f'Total features      : {len(X.columns)}')
print(f'\nListe : {list(X.columns)}')


**Interprétation :**

- `InvStreamDist`, `InvSeaDist` : l'inverse de la distance amplifie l'effet des zones très proches de l'eau — relation non-linéaire que les modèles linéaires seuls ne pourraient pas capturer.
- `water_proximity_score` : agrège les deux sources d'eau en un score unique, en pondérant les cours d'eau davantage (impact plus direct que la mer).
- `topo_wetness` : indice de saturation topographique qui identifie les cuvettes où l'eau s'accumule.
- `flat_lowland`, `precip_low_elev` : variables binaires capturant les conditions les plus à risque.
- Transformées `_log` : réduisent l'asymétrie des distributions de distances, améliorant la convergence des modèles linéaires.

**Signification de `lc_code` (Corine Land Cover) :**

| Code | Type d'occupation | Risque |
|------|-------------------|--------|
| 30 | Zones urbaines | Modéré (sols imperméables) |
| 40 | Zones agricoles | Modéré |
| 60 | Forêts | Faible |
| 80 | Zones humides | Très élevé |
| 90 | Cours d'eau | Extrême |


### 1.4 — Matrice de Corrélation

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1.4 — MATRICE DE CORRÉLATION (deux graphiques complémentaires)
# ═══════════════════════════════════════════════════════════════
# Graphe 1 (sans y) → détection de la multicolinéarité entre features
# Graphe 2 (avec y) → identification des variables les plus prédictives

# Graphe 1 : Heatmap features vs features
plt.figure(figsize=(14, 10))
corr_features = X.corr()
mask = np.triu(np.ones_like(corr_features, dtype=bool))
sns.heatmap(corr_features, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, mask=mask,
            cbar_kws={'shrink': 0.8})
plt.title('Graphe 1 — Corrélation entre Features (sans y)\nBut : détecter la multicolinéarité (> 0.80)', fontsize=12)
plt.tight_layout(); plt.show()

print('Paires fortement corrélées (> 0.80) :')
found = False
for i in range(len(corr_features.columns)):
    for j in range(i+1, len(corr_features.columns)):
        val = abs(corr_features.iloc[i, j])
        if val > 0.80:
            print(f'  ⚠ {corr_features.columns[i]} ↔ {corr_features.columns[j]} = {val:.3f}')
            found = True
if not found:
    print('  ✔ Aucune paire > 0.80 — pas de redondance critique.')

# Graphe 2 : Corrélation avec y
df_with_y = X.copy(); df_with_y['flood'] = y.values
corr_target = (df_with_y.corr()['flood'].drop('flood')
               .sort_values(key=abs, ascending=False))
colors = ['firebrick' if v > 0 else 'steelblue' for v in corr_target.values]
plt.figure(figsize=(12, 6))
plt.barh(corr_target.index[::-1], corr_target.values[::-1], color=colors[::-1], alpha=0.85)
plt.axvline(0, color='black', linewidth=0.8)
plt.axvline( 0.3, color='green', linewidth=1.2, linestyle='--', alpha=0.7, label='Seuil ±0.3')
plt.axvline(-0.3, color='green', linewidth=1.2, linestyle='--', alpha=0.7)
plt.xlabel('Corrélation de Pearson avec flood')
plt.title('Graphe 2 — Corrélation de chaque Feature AVEC y (flood)', fontsize=12)
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

top3_pos = corr_target[corr_target > 0].head(3)
top3_neg = corr_target[corr_target < 0].head(3)
print('\nTop 3 features qui AUGMENTENT le risque :')
for f, v in top3_pos.items(): print(f'  + {f:<30} = {v:+.3f}')
print('Top 3 features qui RÉDUISENT le risque :')
for f, v in top3_neg.items(): print(f'  - {f:<30} = {v:+.3f}')
print('\n⚠ Limite : corrélation de Pearson = relations linéaires uniquement.')
print('  SelectKBest (Étape 2) et importance XGBoost complètent cette analyse.')


**Interprétation :**

- **Graphe 1** : révèle les variables redondantes. Les corrélations entre features originales et leurs versions logarithmiques sont attendues. Si deux features *originales* dépassent 0.80, l'une peut être supprimée, particulièrement pour la Régression Logistique (sensible à la multicolinéarité).

- **Graphe 2** : les barres rouges (corrélation positive) indiquent les variables qui augmentent le risque avec leur valeur (ex. : proximité à l'eau). Les barres bleues (corrélation négative) l'inversent (ex. : altitude élevée, forte pente). Si les features créées par feature engineering apparaissent en tête, cela valide notre ingénierie.


### 1.5 — Pipelines de Prétraitement et Division Train/Test

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1.5 — PIPELINES DE PRÉTRAITEMENT + SPLIT TRAIN/TEST
# ═══════════════════════════════════════════════════════════════
# StandardScaler : x_scaled = (x − μ) / σ → recommandé (robuste aux outliers)
# MinMaxScaler   : x_scaled = (x − min) / (max − min) → comparaison uniquement
#
# ANTI DATA LEAKAGE : les scalers sont fittés UNIQUEMENT sur X_train.
# X_test est transformé avec les statistiques de X_train — jamais re-fitté.

numeric_features     = [c for c in X.columns if c != 'lc_code']
categorical_features = ['lc_code']

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_std = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                      ('scaler',  StandardScaler())]),          numeric_features),
    ('cat', categorical_pipeline,                               categorical_features)
])
preprocessor_mm  = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                      ('scaler',  MinMaxScaler())]),            numeric_features),
    ('cat', categorical_pipeline,                               categorical_features)
])
preprocessor = preprocessor_std   # Scaler par défaut

# ═══════════════════════════════════════════════════════════════
# SPLIT PSEUDO-SPATIAL (anti data-leakage géographique)
# ═══════════════════════════════════════════════════════════════
# ⚠ LIMITE ASSUMÉE : InondatNorthTun.csv ne contient aucune coordonnée
# géographique (lat/lon) — seulement des variables dérivées (distances,
# altitude...). Une vraie validation spatiale (clustering sur lat/lon)
# est donc IMPOSSIBLE avec ce dataset tel quel.
#
# Solution retenue : PSEUDO-SPATIAL CLUSTERING. On regroupe les points
# par PROFIL GÉOGRAPHIQUE (altitude, distance à la mer, au cours d'eau,
# aux routes, aux villes) plutôt que par position exacte. Deux points
# avec un profil géographique quasi-identique sont statistiquement
# proches dans l'espace des features, souvent proches dans la réalité
# — c'est une approximation raisonnable de la proximité spatiale en
# l'absence de coordonnées, mais ce n'est PAS un substitut parfait à
# un vrai découpage géographique. À mentionner explicitement comme
# limite méthodologique dans le rapport.

SPATIAL_PROXY_COLS = ['elev_m', 'STREAM_DIS', 'SEA_DISTAN', 'ROAD_DISTA', 'CITIES_SET']
N_PSEUDO_ZONES     = 12  # nombre de zones pseudo-géographiques (ajustable)

# Imputation (médiane) avant le clustering — KMeans n'accepte pas les NaN.
# Fait ici sur X entier (pré-split, comme l'EDA de l'Étape 1.4) : ne dépend
# pas de y, donc pas de fuite de données liée au split train/test.
proxy_imputed = SimpleImputer(strategy='median').fit_transform(X[SPATIAL_PROXY_COLS])
proxy_scaled  = StandardScaler().fit_transform(proxy_imputed)
kmeans_spatial = KMeans(n_clusters=N_PSEUDO_ZONES, random_state=RANDOM_SEED, n_init=10)
df_raw['pseudo_zone'] = kmeans_spatial.fit_predict(proxy_scaled)

# Visualisation : profil des zones sur 2 axes clés (distance mer / altitude)
plt.figure(figsize=(7, 6))
sc = plt.scatter(X['SEA_DISTAN'], X['elev_m'], c=df_raw['pseudo_zone'],
                  cmap='tab20', s=25, alpha=0.85)
plt.colorbar(sc, label='Pseudo-zone')
plt.xlabel('Distance à la mer (m)'); plt.ylabel('Altitude (m)')
plt.title(f'{N_PSEUDO_ZONES} pseudo-zones géographiques (KMeans sur profil géo)')
plt.tight_layout(); plt.show()

# ── Split : GroupShuffleSplit garantit qu'aucune pseudo-zone n'est dans les deux ensembles ──
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=df_raw['pseudo_zone']))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train    = df_raw['pseudo_zone'].iloc[train_idx].values

print('=' * 55)
print('RÉPARTITION TRAIN / TEST — SPLIT PSEUDO-SPATIAL')
print('=' * 55)
print(f'Zones en Train ({len(set(groups_train))})  : {sorted(set(groups_train))}')
print(f'Zones en Test  ({df_raw["pseudo_zone"].iloc[test_idx].nunique()})   : '
      f'{sorted(set(df_raw["pseudo_zone"].iloc[test_idx]))}')
print(f'Train : {len(X_train):>5} lignes ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test  : {len(X_test):>5} lignes ({len(X_test)/len(X)*100:.0f}%)')
print(f'Ratio flood Train : {y_train.mean()*100:.2f}% | Test : {y_test.mean()*100:.2f}%')
print('✔ Aucune pseudo-zone commune entre Train et Test.')
print('⚠ Approximation : proximité par PROFIL géographique, pas par position exacte.')

# ── CV pseudo-spatiale : chaque fold respecte aussi les pseudo-zones ──
spatial_cv = GroupKFold(n_splits=5)

# ── ANTI DATA LEAKAGE : ré-application du feature engineering à seuils fixes ──
# X_train / X_test contiennent déjà flat_lowland/precip_low_elev calculés (à l'Étape 1.3)
# à partir de quantiles sur TOUT X (utile seulement pour l'EDA/corrélation). On les
# recalcule ici avec des seuils appris UNIQUEMENT sur X_train, et on fige ces seuils
# dans FE_THRESHOLDS pour les réutiliser tels quels sur X_test, sur les nouvelles
# prédictions (Étape 5) et dans Gradio (Étape 6) — jamais recalculés sur test ou 1 point.
FE_THRESHOLDS = {
    'slope_q25':  X_train['slope_d'].quantile(0.25),
    'elev_q25':   X_train['elev_m'].quantile(0.25),
    'precip_q75': X_train['precipmm'].quantile(0.75),
}
X_train = apply_feature_engineering(X_train, thresholds=FE_THRESHOLDS)
X_test  = apply_feature_engineering(X_test,  thresholds=FE_THRESHOLDS)

print('✔ Seuils flat_lowland / precip_low_elev appris sur X_train uniquement (anti data leakage) :')
for k, v in FE_THRESHOLDS.items():
    print(f'   {k} = {v:.4f}')

# Visualisation split
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Train (80%)', 'Test (20%)'], [len(X_train), len(X_test)],
            color=['steelblue', 'firebrick'], alpha=0.85)
axes[0].set_title('Taille des ensembles'); axes[0].set_ylabel('Nombre de points')
for i, v in enumerate([len(X_train), len(X_test)]):
    axes[0].text(i, v+1, str(v), ha='center', fontweight='bold')

w = 0.35
axes[1].bar([0-w/2, 1-w/2], [(y_train==0).sum(), (y_train==1).sum()],
            w, label='Train', color='steelblue', alpha=0.85)
axes[1].bar([0+w/2, 1+w/2], [(y_test==0).sum(), (y_test==1).sum()],
            w, label='Test',  color='firebrick',  alpha=0.85)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['No Flood (0)', 'Flood (1)'])
axes[1].set_title('Stratification vérifiée'); axes[1].legend()
plt.tight_layout(); plt.show()

# Effet des scalers sur STREAM_DIS
vals     = X['STREAM_DIS'].dropna().values
std_vals = StandardScaler().fit_transform(vals.reshape(-1,1)).flatten()
mm_vals  = MinMaxScaler().fit_transform(vals.reshape(-1,1)).flatten()
fig2, ax_arr = plt.subplots(1, 3, figsize=(15, 4))
ax_arr[0].hist(vals,     bins=50, color='steelblue',   alpha=0.8); ax_arr[0].set_title('AVANT — STREAM_DIS')
ax_arr[1].hist(std_vals, bins=50, color='forestgreen', alpha=0.8); ax_arr[1].set_title('StandardScaler → μ=0, σ=1')
ax_arr[2].hist(mm_vals,  bins=50, color='darkorange',  alpha=0.8); ax_arr[2].set_title('MinMaxScaler → [0, 1]')
plt.suptitle('Impact des scalers sur STREAM_DIS', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

pct = (mm_vals < 0.1).mean() * 100
print(f'MinMaxScaler : {pct:.1f}% des valeurs de STREAM_DIS < 0.1 (compression par outlier)')
print('✔ StandardScaler préférable pour ce dataset géographique.')


**Interprétation :** Le split est désormais **pseudo-spatial** (par profil géographique), pas purement aléatoire. Une pseudo-zone entière va soit en train, soit en test — jamais les deux — ce qui limite le risque que le modèle "triche" en s'appuyant sur des points très similaires déjà vus à l'entraînement. ⚠️ Limite assumée : en l'absence de coordonnées lat/lon dans le dataset, cette proximité est estimée à partir du profil géographique (altitude, distances) et non de la position réelle — une vraie validation spatiale nécessiterait des coordonnées géographiques.

Le graphique de comparaison des scalers illustre concrètement le problème de MinMaxScaler : un outlier géographique comprime la majorité des valeurs entre 0 et 0.01, effaçant les différences entre les points. StandardScaler préserve la forme de la distribution tout en ramenant toutes les variables à la même échelle (μ=0, σ=1) — indispensable pour LR et SVM.


### 1.6 — Visualisation AVANT / APRÈS Prétraitement

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1.6 — AVANT / APRÈS PRÉTRAITEMENT
# ═══════════════════════════════════════════════════════════════

feature_names_num = [c for c in X.columns if c != 'lc_code']
X_train_prep = preprocessor.fit_transform(X_train, y_train)
feature_names_cat_after = (preprocessor.named_transformers_['cat']
                            .named_steps['onehot']
                            .get_feature_names_out(categorical_features).tolist())
X_train_prep_df = pd.DataFrame(X_train_prep, columns=feature_names_num + feature_names_cat_after)

vars_to_show = ['elev_m', 'precipmm', 'STREAM_DIS']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, var in enumerate(vars_to_show):
    avant = X_train[var].dropna()
    apres = X_train_prep_df[var]
    axes[0, i].hist(avant, bins=40, color='#378ADD', alpha=0.85, edgecolor='white')
    axes[0, i].set_title(f'AVANT — {var}', fontweight='bold', color='#185FA5')
    axes[0, i].axvline(avant.mean(), color='red', linestyle='--', linewidth=1.5,
                       label=f'μ={avant.mean():.1f}')
    axes[0, i].legend(fontsize=8)
    axes[1, i].hist(apres, bins=40, color='#1D9E75', alpha=0.85, edgecolor='white')
    axes[1, i].set_title(f'APRÈS normalisation — {var}', fontweight='bold', color='#0F6E56')
    axes[1, i].axvline(apres.mean(), color='red', linestyle='--', linewidth=1.5,
                       label=f'μ={apres.mean():.2f}')
    axes[1, i].legend(fontsize=8)
plt.suptitle('Comparaison AVANT / APRÈS Prétraitement (StandardScaler)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

nan_avant = X_train[numeric_features].isnull().sum().sum()
nan_apres = pd.DataFrame(X_train_prep).isnull().sum().sum()
print(f'Valeurs manquantes avant : {nan_avant} | après : {nan_apres}')
print('→ Imputation par la médiane appliquée sur X_train.')
print()
for var in vars_to_show:
    a, b = X_train[var], X_train_prep_df[var]
    print(f'{var}: AVANT μ={a.mean():.2f} σ={a.std():.2f} | APRÈS μ={b.mean():.2f} σ={b.std():.2f}')


**Interprétation :** Avant le StandardScaler, les variables ont des échelles incomparables (`elev_m` de 0 à 900 m, `STREAM_DIS` de 0 à plusieurs dizaines de km). Après normalisation, toutes sont centrées sur μ≈0 avec σ≈1 — le modèle compare les features sur une base équitable. Ce traitement est indispensable pour LR et SVM, qui calculent des distances dans l'espace des features.

## Étape 2 — Sélection de Variables par SelectKBest

La PCA est volontairement exclue : avec moins de 25 features, la réduction de dimension n'est pas nécessaire. De plus, la PCA détruit l'interprétabilité géographique des variables, ce qui est problématique pour l'analyse des résultats. SelectKBest conserve les features originales tout en éliminant les moins informatives.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2.1 — SÉLECTION DE FEATURES : SelectKBest (ANOVA F-score)
# ═══════════════════════════════════════════════════════════════
# ANOVA F-test : mesure si la moyenne de la feature diffère significativement
# entre les classes (flood=0 vs flood=1). Score F élevé → forte discrimination.
#
# ⚠ ANTI DATA LEAKAGE : ajusté sur X_train uniquement (jamais sur X_test).
# Reste une analyse exploratoire : ce SelectKBest n'est PAS branché dans les
# pipelines de modélisation de l'Étape 3 (chaque pipeline utilise toutes les
# features, sélectionnées implicitement par les modèles eux-mêmes).

X_prep_exp = preprocessor.fit_transform(X_train, y_train)  # ajusté sur X_train uniquement (anti leakage)
feature_names_cat_exp = (preprocessor.named_transformers_['cat']
                         .named_steps['onehot']
                         .get_feature_names_out(categorical_features).tolist())
all_feature_names = feature_names_num + feature_names_cat_exp

selector = SelectKBest(score_func=f_classif, k=K_BEST)
selector.fit(X_prep_exp, y_train)
scores_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Score F': selector.scores_,
    'p-value': selector.pvalues_
}).sort_values('Score F', ascending=False)

print(f'Top {K_BEST} features sélectionnées :')
print(scores_df.head(K_BEST).to_string(index=False))

plt.figure(figsize=(12, 5))
top = scores_df.head(K_BEST)
plt.barh(top['Feature'][::-1], top['Score F'][::-1], color='steelblue')
plt.xlabel('Score F (ANOVA)')
plt.title(f'Top {K_BEST} Features — SelectKBest (score élevé = forte discrimination)')
plt.tight_layout(); plt.show()

top3 = scores_df.head(3)['Feature'].tolist()
print(f'\nTop 3 features : {top3}')
eng_feats = ['InvStreamDist', 'water_proximity_score', 'topo_wetness',
             'precip_elev_ratio', 'min_water_dist', 'flat_lowland', 'precip_low_elev']
top_eng = [f for f in scores_df.head(10)['Feature'].tolist() if f in eng_feats]
if top_eng:
    print(f'✔ Features de feature engineering dans le Top 10 : {top_eng}')
else:
    print('→ Les features originales dominent.')


**Interprétation :** SelectKBest identifie les variables dont les distributions sont les plus différentes entre zones inondées et zones sèches. Un Score F élevé et une p-value < 0.05 confirment la significativité statistique. Les features liées à la proximité à l'eau et à l'altitude basse dominent généralement, ce qui est cohérent avec la physique des inondations. Si des features créées par feature engineering apparaissent dans le Top 10, cela valide notre ingénierie de variables.

## Étape 3 — Quatre Modèles Machine Learning

### Modèles et justifications

| Modèle | Principe | Normalisation | Justification du choix |
|--------|----------|---------------|------------------------|
| **Logistic Regression** | Frontière linéaire via sigmoïde | Obligatoire | Baseline interprétable ; coefficients directement lisibles |
| **Random Forest** | Bagging de N arbres indépendants | Non nécessaire | Robuste aux outliers et aux non-linéarités |
| **SVM** | Hyperplan à marge maximale (noyau RBF) | Obligatoire | Performant en dimension élevée |
| **XGBoost** | Boosting séquentiel corrigeant les erreurs | Peu nécessaire | État de l'art sur les données tabulaires |

**Stratégie de validation :** RandomizedSearchCV + StratifiedKFold (k=5) sur X_train. Évaluation finale sur X_test (20%, jamais vu). Critère de sélection : ROC-AUC.


### 3.1 — Fonctions d'Évaluation

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.1 — FONCTIONS D'ÉVALUATION (réutilisées pour chaque modèle)
# ═══════════════════════════════════════════════════════════════

def compute_optimal_threshold_cv(estimator, X_tr, y_tr, cv_splitter, groups=None):
    """Calcule le seuil de décision optimal (critère de Youden : max TPR − FPR)
    à partir de prédictions HORS-ÉCHANTILLON obtenues par validation croisée
    sur X_train/y_train uniquement.

    ⚠ ANTI DATA LEAKAGE : ce seuil ne doit JAMAIS être calculé sur X_test.
    cross_val_predict entraîne un clone de `estimator` sur chaque fold de
    `cv_splitter` et prédit sur le fold laissé de côté — les probabilités
    obtenues sont donc "out-of-fold", jamais vues pendant leur propre
    entraînement, ce qui permet un choix de seuil honnête sans toucher au
    jeu de test.

    `groups` : requis quand cv_splitter est un GroupKFold (validation
    pseudo-spatiale) — garantit que les folds internes respectent eux
    aussi les pseudo-zones géographiques.
    """
    oof_proba = cross_val_predict(estimator, X_tr, y_tr, cv=cv_splitter, groups=groups,
                                   method='predict_proba')[:, 1]
    fpr, tpr, thresholds = roc_curve(y_tr, oof_proba)
    return float(thresholds[np.argmax(tpr - fpr)])


def evaluate_model(y_true, y_proba, model_name='', opt_threshold=None):
    """Calcule 6 métriques pour 3 seuils : Optimal, Default (0.5), High Recall (0.3).

    Parameters
    ----------
    opt_threshold : float optionnel — seuil "Optimal" à utiliser, normalement
        calculé au préalable par `compute_optimal_threshold_cv()` sur X_train.
        Si None, le seuil est recalculé par critère de Youden directement sur
        (y_true, y_proba) passés ici — À ÉVITER quand y_true/y_proba sont
        ceux de X_test, car cela reviendrait à choisir le seuil de décision
        en utilisant les labels du jeu de test (fuite de données).
    """
    y_true, y_proba = np.array(y_true), np.array(y_proba)
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    if opt_threshold is None:
        opt_thr = thresholds[np.argmax(tpr - fpr)]   # Fallback uniquement (ex. usage hors Étape 4)
    else:
        opt_thr = opt_threshold

    results = {}
    for name, thr in [('Optimal', opt_thr), ('Default', 0.5), ('High Recall', 0.3)]:
        yp = (y_proba >= thr).astype(int)
        cm = confusion_matrix(y_true, yp)
        tn, fp, fn, tp = cm.ravel()
        results[name] = {
            'Threshold':   round(float(thr), 3),
            'ROC-AUC':     round(roc_auc_score(y_true, y_proba), 4),
            'Accuracy':    round(accuracy_score(y_true, yp), 4),
            'Recall':      round(recall_score(y_true, yp), 4),
            'Precision':   round(precision_score(y_true, yp, zero_division=0), 4),
            'F1-Score':    round(f1_score(y_true, yp), 4),
            'Specificity': round(tn/(tn+fp) if (tn+fp) > 0 else 0, 4),
            'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
            'Confusion Matrix': cm
        }
    return results


def plot_confusion_matrices(all_res):
    n = len(all_res)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, (name, res) in zip(axes, all_res.items()):
        ConfusionMatrixDisplay(res['Optimal']['Confusion Matrix'],
                               display_labels=['No Flood', 'Flood']).plot(
            ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{name}\nAUC={res["Optimal"]["ROC-AUC"]:.4f}', fontsize=11)
    plt.suptitle('Matrices de Confusion (seuil optimal — critère de Youden, calculé par CV sur X_train)',
                 fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()


def plot_roc_curves(roc_data_list):
    plt.figure(figsize=(8, 6))
    colors = cycle(['steelblue', 'firebrick', 'forestgreen', 'darkorange'])
    for (name, yt, yp), color in zip(roc_data_list, colors):
        fpr, tpr, _ = roc_curve(np.array(yt), np.array(yp))
        plt.plot(fpr, tpr, lw=2, color=color,
                 label=f'{name} (AUC={roc_auc_score(yt, yp):.4f})')
    plt.plot([0,1],[0,1],'k--',lw=1,label='Aléatoire (AUC=0.5)')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title('Courbes ROC — Comparaison des 4 Modèles')
    plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

print('✔ Fonctions d\'évaluation définies (seuil "Optimal" désormais calculé par CV sur X_train).')


**Fonctions définies :** `evaluate_model()` calcule 6 métriques pour trois seuils de décision. Le **seuil optimal** (critère de Youden, maximise TPR − FPR) est désormais calculé par `compute_optimal_threshold_cv()` **uniquement sur X_train**, via des prédictions hors-échantillon obtenues par validation croisée (`cross_val_predict`) — jamais sur X_test, afin d'éviter toute fuite dans le choix du seuil de décision (voir Étape 3.3). Le **seuil High Recall (0.3)** minimise les inondations manquées, adapté à un système d'alerte précoce où manquer une inondation est plus coûteux qu'une fausse alarme.

### 3.2 — Définition des 4 Modèles et Hyperparamètres

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.2 — 4 MODÈLES ET GRILLES D'HYPERPARAMÈTRES
# ═══════════════════════════════════════════════════════════════
# RandomizedSearchCV vs GridSearchCV : tire n_iter combinaisons aléatoires
# → 40× plus rapide que GridSearchCV pour résultats comparables (Bergstra & Bengio, 2012).
# StratifiedKFold k=5 : ratio de classes identique dans chaque fold.

# cv pseudo-spatiale (GroupKFold) : chaque fold respecte les pseudo-zones
# géographiques définies à l'Étape 1.5 — remplace le StratifiedKFold classique,
# qui mélangeait des points au profil géographique proche entre folds train/val.
cv = spatial_cv

# MODÈLE 1 — Logistic Regression
# C : régularisation inverse | L1 : sélection implicite | L2 : réduction uniforme
lr_params = {'model__C': uniform(0.001, 100), 'model__penalty': ['l1', 'l2'],
             'model__solver': ['liblinear', 'saga'], 'model__max_iter': [3000]}
lr_pipe_std = Pipeline([('preprocessor', preprocessor_std),
                        ('model', LogisticRegression(random_state=RANDOM_SEED))])
lr_pipe_mm  = Pipeline([('preprocessor', preprocessor_mm),
                        ('model', LogisticRegression(random_state=RANDOM_SEED))])

# MODÈLE 2 — Random Forest
# max_depth borné : évite l'overfitting parfait (max_depth=None)
rf_params = {'model__n_estimators': randint(100, 400), 'model__max_depth': [5, 8, 10, 15, 20],
             'model__min_samples_split': randint(2, 20), 'model__min_samples_leaf': randint(1, 10),
             'model__max_features': ['sqrt', 'log2'], 'model__class_weight': ['balanced', None]}
rf_pipe_std = Pipeline([('preprocessor', preprocessor_std),
                        ('model', RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1))])
rf_pipe_mm  = Pipeline([('preprocessor', preprocessor_mm),
                        ('model', RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1))])

# MODÈLE 3 — SVM (le plus sensible aux échelles → StandardScaler obligatoire)
# Grand gamma = frontière très locale = risque d'overfitting
svm_params = {'model__C': uniform(0.1, 50), 'model__gamma': ['scale', 'auto'],
              'model__kernel': ['rbf', 'sigmoid']}
svm_pipe_std = Pipeline([('preprocessor', preprocessor_std),
                         ('model', SVC(probability=True, random_state=RANDOM_SEED))])
svm_pipe_mm  = Pipeline([('preprocessor', preprocessor_mm),
                         ('model', SVC(probability=True, random_state=RANDOM_SEED))])

# MODÈLE 4 — XGBoost
# subsample + colsample_bytree : régularisation stochastique
# scale_pos_weight : pondération de la classe minoritaire
xgb_params = {'model__n_estimators': randint(100, 500), 'model__max_depth': [3, 4, 5, 6, 8],
              'model__learning_rate': uniform(0.01, 0.3), 'model__subsample': uniform(0.6, 0.4),
              'model__colsample_bytree': uniform(0.6, 0.4), 'model__reg_alpha': uniform(0, 1),
              'model__reg_lambda': uniform(1, 3),
              'model__scale_pos_weight': [1, y_train.value_counts()[0] / y_train.value_counts()[1]]}
xgb_pipe_std = Pipeline([('preprocessor', preprocessor_std),
                          ('model', XGBClassifier(random_state=RANDOM_SEED, n_jobs=-1,
                                                  eval_metric='logloss', verbosity=0))])
xgb_pipe_mm  = Pipeline([('preprocessor', preprocessor_mm),
                          ('model', XGBClassifier(random_state=RANDOM_SEED, n_jobs=-1,
                                                  eval_metric='logloss', verbosity=0))])

models_config = {
    'Logistic Regression': (lr_pipe_std,  lr_pipe_mm,  lr_params,  20),
    'Random Forest':       (rf_pipe_std,  rf_pipe_mm,  rf_params,  30),
    'SVM':                 (svm_pipe_std, svm_pipe_mm, svm_params, 15),
    'XGBoost':             (xgb_pipe_std, xgb_pipe_mm, xgb_params, 30),
}
print('✔ 4 modèles configurés :')
for name, (_, __, ___, n) in models_config.items():
    print(f'  • {name:<22} ({n} itérations RandomizedSearchCV)')


**Justification des hyperparamètres :**

- **LR** : `C` petit = forte régularisation (modèle général). `L1` met certains coefficients à zéro (sélection implicite) ; `L2` les réduit uniformément.
- **RF** : `max_depth` borné à [5, 20] — la valeur `None` crée des arbres qui mémorisent parfaitement le train set. `min_samples_leaf` lisse les prédictions.
- **SVM** : `C` et `gamma` interagissent — grand `C` + grand `gamma` = frontière très irrégulière = overfitting. RandomizedSearchCV explore efficacement cet espace.
- **XGBoost** : `learning_rate` faible + `n_estimators` élevé = apprentissage plus précis. `subsample` et `colsample_bytree` sont des formes de régularisation stochastique.


### 3.3 — Entraînement, Optimisation et Comparaison des Scalers

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.3 — ENTRAÎNEMENT ET COMPARAISON DES SCALERS
# ═══════════════════════════════════════════════════════════════
# Pour chaque modèle : entraînement avec StandardScaler ET MinMaxScaler.
#
# ⚠ ANTI DATA LEAKAGE : le meilleur scaler ET le meilleur modèle global sont
# sélectionnés sur le score de VALIDATION CROISÉE (search.best_score_, calculé
# uniquement sur X_train), et non sur l'AUC de X_test. X_test n'est utilisé
# qu'une seule fois, à la toute fin, pour le rapport final — jamais pour
# prendre une décision de sélection. Le comparer/choisir sur le test
# transformerait implicitement X_test en un second jeu de validation, ce qui
# biaiserait de façon optimiste l'évaluation finale.

best_estimators   = {}
all_results       = {}
roc_data_list     = []
scaler_comparison = []
overfitting_log   = []

for name, (pipe_std, pipe_mm, params, n_iter) in models_config.items():
    print(f'\n{"=" * 60}')
    print(f'  {name}  ({n_iter} itérations × 2 scalers)')
    print('=' * 60)

    best_cv_auc, best_pipe_final, best_scaler_name, best_params = -1, None, '', None

    for scaler_name, pipe in [('StandardScaler', pipe_std), ('MinMaxScaler', pipe_mm)]:
        print(f'  ── {scaler_name} ──')
        search = RandomizedSearchCV(
            pipe, params, n_iter=n_iter, cv=cv, scoring='roc_auc',
            n_jobs=-1, random_state=RANDOM_SEED, verbose=0, refit=True
        )
        search.fit(X_train, y_train, groups=groups_train)  # GroupKFold exige les groupes pseudo-spatiaux

        # AUC test calculé uniquement pour reporting/monitoring — jamais pour décider.
        yp_test  = search.best_estimator_.predict_proba(X_test)[:, 1]
        yp_train = search.best_estimator_.predict_proba(X_train)[:, 1]
        auc_test  = roc_auc_score(y_test,  yp_test)
        auc_train = roc_auc_score(y_train, yp_train)
        acc_test  = accuracy_score(y_test, search.best_estimator_.predict(X_test))

        print(f'     AUC CV   : {search.best_score_:.4f}  ← critère de sélection')
        print(f'     AUC Test : {auc_test:.4f} | Acc Test : {acc_test:.4f}  (reporting uniquement)')

        scaler_comparison.append({
            'Modèle': name, 'Scaler': scaler_name,
            'AUC CV': round(search.best_score_, 4),
            'AUC Train': round(auc_train, 4), 'AUC Test': round(auc_test, 4),
            'Gap': round(auc_train - auc_test, 4)
        })

        # ── Sélection du scaler : sur le score de CV, PAS sur auc_test ──
        if search.best_score_ > best_cv_auc:
            best_cv_auc, best_pipe_final = search.best_score_, search.best_estimator_
            best_scaler_name = scaler_name
            best_params = search.best_params_

    best_estimators[name] = best_pipe_final
    yp_best  = best_pipe_final.predict_proba(X_test)[:, 1]
    yp_tr    = best_pipe_final.predict_proba(X_train)[:, 1]
    auc_test_best = roc_auc_score(y_test, yp_best)
    gap      = roc_auc_score(y_train, yp_tr) - auc_test_best

    print(f'\n  ✔ Meilleur scaler : {best_scaler_name} (AUC CV={best_cv_auc:.4f}, AUC Test={auc_test_best:.4f})')
    print(f'  Hyperparamètres retenus :')
    for p, v in best_params.items(): print(f'    {p}: {v}')
    status = '✔ OK' if gap <= 0.05 else ('⚠ Léger overfitting' if gap <= 0.10 else '⚠ Overfitting')
    print(f'  Gap Train-Test : {gap:.4f}  {status}')

    overfitting_log.append({
        'Modèle': name, 'Scaler': best_scaler_name,
        'AUC CV':    round(best_cv_auc, 4),
        'AUC Train': round(roc_auc_score(y_train, yp_tr), 4),
        'AUC Test':  round(auc_test_best, 4),
        'Gap':       round(gap, 4),
        'Acc Train': round(accuracy_score(y_train, best_pipe_final.predict(X_train)), 4),
        'Acc Test':  round(accuracy_score(y_test,  best_pipe_final.predict(X_test)), 4)
    })
    # ── ANTI DATA LEAKAGE : seuil "Optimal" calculé par CV sur X_train uniquement ──
    # (jamais sur y_test/yp_best) — cf. compute_optimal_threshold_cv() en 3.1
    opt_thr_cv = compute_optimal_threshold_cv(best_pipe_final, X_train, y_train, cv, groups=groups_train)
    print(f'  Seuil optimal (Youden, calculé par CV sur X_train) : {opt_thr_cv:.3f}')

    all_results[name] = evaluate_model(y_test, yp_best, name, opt_threshold=opt_thr_cv)
    roc_data_list.append((name, y_test, yp_best))

print('\n✔ Entraînement terminé.')
print('\nComparaison StandardScaler vs MinMaxScaler :')
print(pd.DataFrame(scaler_comparison).to_string(index=False))
print('\nSurveillance Overfitting :')
ovf_df = pd.DataFrame(overfitting_log)
print(ovf_df.to_string(index=False))

# ── Sélection du meilleur modèle global : sur AUC CV, PAS sur AUC Test ──
best_global_name = ovf_df.sort_values('AUC CV', ascending=False).iloc[0]['Modèle']
MODEL_PATH = '/content/drive/MyDrive/PFA_ML/MODELS/best_model.joblib' if COLAB else 'best_model.joblib'
if COLAB: os.makedirs('/content/drive/MyDrive/PFA_ML/MODELS', exist_ok=True)
joblib.dump(best_estimators[best_global_name], MODEL_PATH)
print(f'\n💾 Meilleur modèle (sélectionné sur AUC CV) : {best_global_name}  →  {MODEL_PATH}')


**Interprétation :** Pour chaque algorithme, deux versions sont comparées. Le scaler et le modèle retenus sont ceux qui obtiennent le meilleur **score moyen de validation croisée (AUC CV)** sur X_train — X_test n'intervient à aucun moment dans cette décision, il n'est utilisé qu'une seule fois à la fin pour le rapport final. Les résultats confirment les propriétés théoriques : LR et SVM montrent un écart significatif entre les scalers (sensibles aux échelles), tandis que RF et XGBoost affichent un écart quasi-nul (arbres de décision insensibles aux échelles). Le tableau d'overfitting signale les modèles dont le gap AUC train-test dépasse 0.05.

### 3.4 — Visualisation : Impact du Scaler par Modèle

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3.4 — GRAPHIQUE COMPARATIF : StandardScaler vs MinMaxScaler
# ═══════════════════════════════════════════════════════════════

sc_df   = pd.DataFrame(scaler_comparison)
modeles = sc_df['Modèle'].unique()
x, w = np.arange(len(modeles)), 0.35

std_aucs = sc_df[sc_df['Scaler']=='StandardScaler']['AUC Test'].values
mm_aucs  = sc_df[sc_df['Scaler']=='MinMaxScaler']['AUC Test'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].bar(x - w/2, std_aucs, w, label='StandardScaler', color='steelblue',  alpha=0.85)
bars2 = axes[0].bar(x + w/2, mm_aucs,  w, label='MinMaxScaler',   color='darkorange', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(modeles, rotation=10, ha='right')
axes[0].set_ylim(max(min(std_aucs.min(), mm_aucs.min()) - 0.05, 0), 1.02)
axes[0].set_ylabel('ROC-AUC (Test)'); axes[0].set_title('Impact du Scaler sur ROC-AUC')
axes[0].legend()
for bar in list(bars1) + list(bars2):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{bar.get_height():.3f}', ha='center', fontsize=8)

diff = std_aucs - mm_aucs
colors_diff = ['steelblue' if d >= 0 else 'firebrick' for d in diff]
axes[1].bar(modeles, diff, color=colors_diff, alpha=0.85)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_ylabel('Δ AUC = Standard − MinMax')
axes[1].set_title('Avantage StandardScaler vs MinMaxScaler')
for i, d in enumerate(diff):
    axes[1].text(i, d + (0.001 if d >= 0 else -0.003),
                 f'{d:+.4f}', ha='center', va='bottom' if d >= 0 else 'top', fontsize=9)
plt.suptitle('Comparaison des Scalers — Impact sur les 4 Modèles', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('LR et SVM : écart attendu significatif (sensibles aux échelles).')
print('RF et XGBoost : écart attendu faible (arbres, insensibles aux échelles).')


## Étape 4 — Évaluation Complète des Modèles

Évaluation sur X_test (20%, jamais vus pendant l'entraînement) avec 6 métriques :

| Métrique | Formule | Ce qu'elle mesure |
|----------|---------|-------------------|
| **ROC-AUC** | Aire sous la courbe ROC | Performance globale, robuste au déséquilibre |
| **Accuracy** | (TP+TN)/Total | Proportion de prédictions correctes |
| **Recall** | TP/(TP+FN) | Fraction d'inondations détectées (**priorité absolue**) |
| **Precision** | TP/(TP+FP) | Fraction de vraies alertes parmi les alertes émises |
| **F1-Score** | 2×P×R/(P+R) | Compromis Precision/Recall |
| **Specificity** | TN/(TN+FP) | Fraction de zones sèches correctement identifiées |


### 4.1 — Tableau Comparatif et Graphiques de Performance

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.1 — TABLEAU COMPARATIF DES 4 MODÈLES
# ═══════════════════════════════════════════════════════════════

rows = []
for mname, tres in all_results.items():
    opt = tres['Optimal']
    rows.append({'Modèle': mname, 'ROC-AUC': opt['ROC-AUC'], 'Accuracy': opt['Accuracy'],
                 'Recall': opt['Recall'], 'Precision': opt['Precision'],
                 'F1-Score': opt['F1-Score'], 'Specificity': opt['Specificity'],
                 'Seuil opt.': opt['Threshold']})

summary_df = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('=' * 70)
print('RÉSULTATS SUR X_TEST (données jamais vues pendant l\'entraînement)')
print('=' * 70)
print(summary_df.to_string(index=False))
best = summary_df.iloc[0]
print(f'\n🏆 Meilleur modèle : {best["Modèle"]} (AUC = {best["ROC-AUC"]:.4f})')

# Graphique comparatif des métriques
metrics_plot = ['ROC-AUC', 'Accuracy', 'Recall', 'F1-Score', 'Precision']
x = np.arange(len(summary_df)); width = 0.15
colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']
fig, ax = plt.subplots(figsize=(11, 5))
for i, (met, col) in enumerate(zip(metrics_plot, colors_bar)):
    ax.bar(x + i*width, summary_df[met], width, label=met, color=col, alpha=0.85)
ax.set_xticks(x + width*2); ax.set_xticklabels(summary_df['Modèle'])
ax.set_ylim(0.5, 1.02); ax.set_ylabel('Score')
ax.set_title('Comparaison des 4 Modèles — Toutes les Métriques (seuil optimal)')
ax.legend(loc='lower right', fontsize=9)
ax.axhline(0.9, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()

# Graphique overfitting
ovf_df = pd.DataFrame(overfitting_log)
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
x2, w2 = np.arange(len(ovf_df)), 0.35
axes2[0].bar(x2 - w2/2, ovf_df['AUC Train'], w2, label='Train', color='steelblue', alpha=0.85)
axes2[0].bar(x2 + w2/2, ovf_df['AUC Test'],  w2, label='Test',  color='firebrick', alpha=0.85)
axes2[0].set_xticks(x2); axes2[0].set_xticklabels(ovf_df['Modèle'], rotation=10)
axes2[0].set_ylim(0.5, 1.05); axes2[0].set_ylabel('ROC-AUC')
axes2[0].set_title('AUC : Train vs Test'); axes2[0].legend()
colors_gap = ['green' if g <= 0.05 else 'orangered' for g in ovf_df['Gap']]
bars = axes2[1].bar(ovf_df['Modèle'], ovf_df['Gap'], color=colors_gap, alpha=0.85)
axes2[1].axhline(0.05, color='red', linestyle='--', linewidth=2, label='Seuil 0.05')
axes2[1].set_ylabel('AUC Train − AUC Test')
axes2[1].set_title('Indicateur d\'Overfitting (vert = OK)')
axes2[1].legend()
for bar, v in zip(bars, ovf_df['Gap']):
    axes2[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                  f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.suptitle('Surveillance Overfitting', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


**Interprétation :** Le classement par ROC-AUC est la référence principale (insensible au déséquilibre de classes). XGBoost et Random Forest surpassent généralement LR et SVM sur les données tabulaires grâce à leur capacité à modéliser des frontières non-linéaires. Le graphique d'overfitting identifie les modèles dont le gap AUC train-test dépasse 0.05, indiquant une mémorisation du train set. Le seuil « Optimal » utilisé pour Accuracy/Recall/Precision/F1/Specificity dans ce tableau est désormais calculé par validation croisée sur X_train (voir Étape 3.1/3.3) — X_test n'intervient à aucun moment dans son choix.

### 4.2 — Matrices de Confusion

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.2 — MATRICES DE CONFUSION
# ═══════════════════════════════════════════════════════════════
# TP : inondé → prédit inondé  ✔    TN : sec → prédit sec        ✔
# FP : sec    → prédit inondé  ✗    FN : inondé → prédit sec     ✗ ← DANGER

plot_confusion_matrices(all_results)

print('Taux de Faux Négatifs par modèle (inondations non détectées) :')
for name, res in all_results.items():
    opt = res['Optimal']
    denom = opt['TP'] + opt['FN']
    taux_fn = opt['FN'] / denom if denom > 0 else 0
    print(f'  {name:<22}  FN={opt["FN"]:>4}  Taux FN={taux_fn*100:.1f}%')


**Interprétation :** Dans un contexte de prévention des risques naturels, **minimiser les Faux Négatifs (FN) est la priorité absolue** : une inondation non détectée peut mettre des vies en danger. Le seuil de décision peut être abaissé à 0.3 (colonne « High Recall ») pour maximiser le Recall, acceptant davantage de fausses alarmes en échange d'une meilleure détection des cas critiques.

### 4.3 — Courbes ROC et Analyse par Seuils

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.3 — COURBES ROC ET ANALYSE PAR SEUILS
# ═══════════════════════════════════════════════════════════════

plot_roc_curves(roc_data_list)

best_model_name = summary_df.iloc[0]['Modèle']
print(f'Analyse des seuils — {best_model_name} :')
thr_rows = []
for tname, metrics in all_results[best_model_name].items():
    thr_rows.append({'Seuil': tname, 'Valeur': metrics['Threshold'],
                     'Recall': metrics['Recall'], 'Precision': metrics['Precision'],
                     'F1': metrics['F1-Score'], 'FN': metrics['FN']})
print(pd.DataFrame(thr_rows).to_string(index=False))
print()
print('→ Seuil Optimal   : meilleur équilibre Recall/Spécificité (Youden).')
print('→ High Recall=0.3 : maximise la détection, minimise les FN. Recommandé pour alertes précoces.')
print('→ Default=0.5     : seuil standard, bon compromis général.')


**Interprétation :** La courbe ROC trace le Taux de Vrais Positifs (Recall) en fonction du Taux de Faux Positifs pour tous les seuils. L'AUC synthétise la performance en un seul chiffre : 1.0 = parfait, 0.5 = aléatoire. Plus la courbe est proche du coin supérieur gauche, meilleur est le modèle. Le tableau des seuils montre le compromis Recall/Precision : abaisser le seuil augmente le Recall (moins de FN) mais réduit la Precision (plus de fausses alarmes). Le seuil « Optimal » de ce tableau est fixé au préalable par validation croisée sur X_train, pas recalculé sur X_test.

### 4.4 — Importance des Features : Random Forest et XGBoost

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4.4 — IMPORTANCE DES FEATURES (RF et XGBoost)
# ═══════════════════════════════════════════════════════════════
# RF  : importance par réduction d'impureté de Gini (moyenne sur tous les arbres)
# XGB : importance par gain (amélioration de la perte à chaque split)

def get_feature_names(pipe_best, feat_num, cat_feats):
    feat_cat = (pipe_best.named_steps['preprocessor']
                .named_transformers_['cat']
                .named_steps['onehot']
                .get_feature_names_out(cat_feats).tolist())
    return feat_num + feat_cat

if 'Random Forest' in best_estimators and 'XGBoost' in best_estimators:
    rf_pipe_b  = best_estimators['Random Forest']
    xgb_pipe_b = best_estimators['XGBoost']
    feats_rf   = get_feature_names(rf_pipe_b,  feature_names_num, categorical_features)
    feats_xgb  = get_feature_names(xgb_pipe_b, feature_names_num, categorical_features)

    imp_rf  = pd.DataFrame({'Feature': feats_rf,  'RF':  rf_pipe_b.named_steps['model'].feature_importances_})
    imp_xgb = pd.DataFrame({'Feature': feats_xgb, 'XGB': xgb_pipe_b.named_steps['model'].feature_importances_})
    imp_rf  = imp_rf.sort_values('RF',  ascending=False).head(20)
    imp_xgb = imp_xgb.sort_values('XGB', ascending=False).head(20)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    colors_rf = ['#1565C0' if i < 5 else '#64B5F6' for i in range(len(imp_rf))]
    axes[0].barh(imp_rf['Feature'][::-1], imp_rf['RF'][::-1], color=colors_rf[::-1], alpha=0.9)
    axes[0].set_xlabel('Importance (Gini)'); axes[0].set_title('Top 20 Features — Random Forest')

    colors_xgb = ['#E65100' if i < 5 else '#FFAB76' for i in range(len(imp_xgb))]
    axes[1].barh(imp_xgb['Feature'][::-1], imp_xgb['XGB'][::-1], color=colors_xgb[::-1], alpha=0.9)
    axes[1].set_xlabel('Importance (Gain)'); axes[1].set_title('Top 20 Features — XGBoost')

    plt.suptitle('Importance des Features : RF vs XGBoost', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    print('Top 10 — Random Forest :')
    print(imp_rf.head(10).to_string(index=False))
    print('\nTop 10 — XGBoost :')
    print(imp_xgb.head(10).to_string(index=False))

    eng_feats = ['InvStreamDist', 'water_proximity_score', 'topo_wetness',
                 'precip_elev_ratio', 'min_water_dist', 'flat_lowland', 'precip_low_elev']
    top_eng_rf  = [f for f in imp_rf.head(10)['Feature'].tolist() if f in eng_feats]
    top_eng_xgb = [f for f in imp_xgb.head(10)['Feature'].tolist() if f in eng_feats]
    print(f'\nFeatures de feature engineering dans le Top 10 RF  : {top_eng_rf}')
    print(f'Features de feature engineering dans le Top 10 XGB : {top_eng_xgb}')
else:
    print('⚠ Exécuter d\'abord la cellule 3.3.')


**Interprétation :** Les deux méthodes d'importance convergent généralement vers les mêmes variables clés (proximité à l'eau, altitude, précipitations), ce qui renforce la robustesse de l'interprétation. RF utilise la réduction d'impureté de Gini ; XGBoost utilise le gain de performance à chaque split. Si des features créées par feature engineering apparaissent dans les deux classements, cela confirme leur valeur ajoutée.

## Étape 5 — Prédiction sur Nouvelles Données

Le pipeline sauvegardé est utilisé pour prédire le risque d'inondation sur des points géographiques inédits. Il suffit de fournir les données brutes — le pipeline applique automatiquement le même prétraitement qu'à l'entraînement.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5.0 — PRÉDICTION SUR NOUVELLES DONNÉES
# ═══════════════════════════════════════════════════════════════

try:
    model_for_demo = joblib.load(MODEL_PATH)
    print(f'✔ Modèle rechargé depuis : {MODEL_PATH}')
except:
    model_for_demo = best_estimators[best_global_name]
    print('✔ Modèle utilisé depuis la mémoire.')

# CAS 1 — Un point géographique unique
print('\n' + '─'*55)
print('CAS 1 — Point unique : basse plaine côtière près de Bizerte')
print('─'*55)

nouveau_point = pd.DataFrame([{
    'elev_m': 12, 'slope_d': 0.8, 'precipmm': 185,
    'STREAM_DIS': 80, 'SEA_DISTAN': 1200, 'ROAD_DISTA': 500,
    'CITIES_SET': 3000, 'lc_code': 80
}])
nouveau_point = apply_feature_engineering(nouveau_point, thresholds=FE_THRESHOLDS)  # seuils appris sur X_train

proba_flood = model_for_demo.predict_proba(nouveau_point)[0, 1]
barre = '█' * int(proba_flood * 30) + '░' * (30 - int(proba_flood * 30))
print(f'  Altitude : 12 m | Pente : 0.8° | Précipitations : 185 mm | Dist. eau : 80 m')
print(f'  Probabilité : |{barre}| {proba_flood*100:.1f}%')
if proba_flood >= 0.70:   print('  Décision : 🔴 RISQUE ÉLEVÉ — ALERTE INONDATION')
elif proba_flood >= 0.50: print('  Décision : 🟠 RISQUE MODÉRÉ — Surveillance recommandée')
elif proba_flood >= 0.30: print('  Décision : 🟡 RISQUE FAIBLE')
else:                     print('  Décision : 🟢 RISQUE TRÈS FAIBLE')

# CAS 2 — Plusieurs points (CSV)
print('\n' + '─'*55)
print('CAS 2 — Cinq points géographiques variés')
print('─'*55)

nouveaux_points = pd.DataFrame([
    {'elev_m':  8, 'slope_d': 0.5, 'precipmm': 200, 'STREAM_DIS':  50, 'SEA_DISTAN':  800, 'ROAD_DISTA': 300,  'CITIES_SET': 2000,  'lc_code': 90},
    {'elev_m': 45, 'slope_d': 3.2, 'precipmm': 110, 'STREAM_DIS': 800, 'SEA_DISTAN': 8000, 'ROAD_DISTA': 1500, 'CITIES_SET': 12000, 'lc_code': 40},
    {'elev_m':120, 'slope_d': 8.1, 'precipmm':  90, 'STREAM_DIS':3000, 'SEA_DISTAN':25000, 'ROAD_DISTA': 5000, 'CITIES_SET': 30000, 'lc_code': 60},
    {'elev_m': 18, 'slope_d': 1.1, 'precipmm': 160, 'STREAM_DIS': 150, 'SEA_DISTAN': 2500, 'ROAD_DISTA': 400,  'CITIES_SET': 5000,  'lc_code': 80},
    {'elev_m':250, 'slope_d':12.5, 'precipmm':  70, 'STREAM_DIS':5000, 'SEA_DISTAN':60000, 'ROAD_DISTA':10000, 'CITIES_SET': 45000, 'lc_code': 60},
])
df_new = apply_feature_engineering(nouveaux_points.copy(), thresholds=FE_THRESHOLDS)  # mêmes seuils que l'entraînement
probas = model_for_demo.predict_proba(df_new)[:, 1]
labels = ['🔴 ÉLEVÉ' if p >= 0.70 else '🟠 MODÉRÉ' if p >= 0.50 else '🟡 FAIBLE' if p >= 0.30 else '🟢 TRÈS FAIBLE'
          for p in probas]

res = pd.DataFrame({
    'Point': [f'Point {i+1}' for i in range(len(df_new))],
    'Altitude(m)': nouveaux_points['elev_m'].values,
    'Précip(mm)': nouveaux_points['precipmm'].values,
    'Dist_eau(m)': nouveaux_points['STREAM_DIS'].values,
    'lc_code': nouveaux_points['lc_code'].values,
    'P(flood)': [f'{p*100:.1f}%' for p in probas],
    'Risque': labels
})
print(res.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
couleurs = ['firebrick' if p >= 0.70 else 'darkorange' if p >= 0.50 else 'gold' if p >= 0.30 else 'forestgreen'
            for p in probas]
bars = ax.bar(res['Point'], probas * 100, color=couleurs, alpha=0.85, edgecolor='white')
ax.axhline(70, color='red',    linestyle='--', linewidth=1.5, label='Seuil alerte (70%)')
ax.axhline(50, color='orange', linestyle='--', linewidth=1.5, label='Seuil modéré (50%)')
for bar, p in zip(bars, probas):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{p*100:.1f}%', ha='center', fontweight='bold', fontsize=10)
ax.set_ylabel('Probabilité d\'inondation (%)'); ax.set_ylim(0, 110)
ax.set_title('Prédiction du Risque — Nouveaux Points Géographiques')
ax.legend(loc='upper right', fontsize=9); plt.tight_layout(); plt.show()


**Interprétation :** Le pipeline rechargé depuis le fichier `.joblib` applique automatiquement les mêmes transformations (imputation, normalisation, encodage) qu'à l'entraînement — avec les statistiques apprises sur X_train. La sortie est une probabilité entre 0 et 1, convertie en niveau d'alerte. Le seuil peut être adapté au contexte : 0.30 pour un système d'alerte précoce (priorité au Recall), 0.50 pour un rapport standard, 0.70 pour une décision d'évacuation.

**Remarque anti-fuite / anti-bug :** `apply_feature_engineering()` reçoit ici `thresholds=FE_THRESHOLDS`, les seuils appris sur X_train à l'Étape 1.5. Cela corrige un bug de la version précédente où un point unique recevait toujours `flat_lowland=0` et `precip_low_elev=0` (un quantile calculé sur une seule ligne n'a pas de sens) et garantit que les prédictions en production utilisent exactement les mêmes seuils qu'à l'entraînement.

## Étape 6 — Interface Utilisateur Interactive (Gradio)

Interface web permettant à un utilisateur non technique de tester le modèle sans écrire de code. Elle affiche les prédictions des quatre modèles, la décision par vote, et les métriques d'évaluation.

**Comment tester :** saisir les caractéristiques d'un point géographique via les curseurs → cliquer sur « Prédire » → consulter les résultats dans les deux onglets.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6.1 — CHARGEMENT DU MODÈLE ET VÉRIFICATION DE GRADIO
# ═══════════════════════════════════════════════════════════════
# Cette cellule recharge le modèle depuis Drive avant de lancer Gradio.
# Si la session Colab redémarre, best_estimators{} est vide.
# Le fichier .joblib garantit la disponibilité du modèle.

import joblib

try:
    model_for_gradio = joblib.load(MODEL_PATH)
    print(f'✔ Modèle chargé depuis : {MODEL_PATH}')
except Exception as e:
    if 'best_estimators' in dir() and len(best_estimators) > 0:
        preferred = 'XGBoost' if 'XGBoost' in best_estimators else list(best_estimators.keys())[0]
        model_for_gradio = best_estimators[preferred]
        print(f'✔ Modèle utilisé depuis la mémoire : {preferred}')
    else:
        model_for_gradio = None
        print('❌ Aucun modèle disponible — exécuter d\'abord la cellule 3.3 !')

try:
    import gradio as gr
    GRADIO_OK = True
    print(f'✔ Gradio disponible (version {gr.__version__})')
except ImportError:
    GRADIO_OK = False
    print('Gradio non installé. Exécuter : !pip install gradio')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6.2 — INTERFACE GRADIO
# ═══════════════════════════════════════════════════════════════

if GRADIO_OK and model_for_gradio is not None:

    def predict_and_metrics(elev_m, slope_d, precipmm, stream_dis,
                            sea_distan, road_dista, cities_set, lc_code_val):
        """
        Fonction principale appelée à chaque clic sur le bouton 'Prédire'.
        Retourne les prédictions des 4 modèles et les métriques d'évaluation.
        """
        row = pd.DataFrame([{
            'elev_m': elev_m, 'slope_d': slope_d, 'precipmm': precipmm,
            'STREAM_DIS': stream_dis, 'SEA_DISTAN': sea_distan,
            'ROAD_DISTA': road_dista, 'CITIES_SET': cities_set,
            'lc_code': lc_code_val
        }])
        row = apply_feature_engineering(row, thresholds=FE_THRESHOLDS)  # mêmes seuils que l'entraînement (anti leakage + fix bug seuil sur 1 ligne)

        # Prédictions de chaque modèle
        lines_pred = ['## Résultats de Prédiction\n']
        probas = {}
        for mname, estimator in best_estimators.items():
            prob = float(estimator.predict_proba(row)[0, 1])
            probas[mname] = prob
            dec = '🔴 INONDÉ' if prob >= 0.5 else '🟢 Non inondé'
            bar = '█' * int(prob*20) + '░' * (20 - int(prob*20))
            lines_pred += [f'**{mname}**',
                           f'Probabilité : `{prob:.3f}` |{bar}|',
                           f'Décision : {dec}\n']

        mean_prob = float(np.mean(list(probas.values())))
        decision  = '🔴 RISQUE ÉLEVÉ — INONDATION PROBABLE' if mean_prob >= 0.5 else '🟢 RISQUE FAIBLE'
        lines_pred += ['---',
                       f'**Probabilité moyenne (4 modèles) : {mean_prob:.3f}**',
                       f'**Décision finale : {decision}**']

        # Métriques sur X_test
        lines_met = ['## Métriques d\'Évaluation (sur X_test)\n',
                     '| Modèle | ROC-AUC | Accuracy | Recall | Precision | F1-Score |',
                     '|--------|---------|----------|--------|-----------|----------|']
        for mname, res in all_results.items():
            opt = res['Optimal']
            lines_met.append(f"| {mname} | {opt['ROC-AUC']} | {opt['Accuracy']} "
                             f"| {opt['Recall']} | {opt['Precision']} | {opt['F1-Score']} |")
        lines_met.append('\n*Métriques calculées sur 20% des données de test.*')

        return '\n'.join(lines_pred), '\n'.join(lines_met)

    # Construction de l'interface
    with gr.Blocks(title='Prédiction Inondations — Nord Tunisie',
                   theme=gr.themes.Soft()) as demo:

        gr.Markdown('# 🌊 Prédiction des Inondations — Nord Tunisie')
        gr.Markdown('**PFA Machine Learning — Université iTeam 2025/2026**')
        gr.Markdown('*Saisissez les caractéristiques géographiques d\'un point pour estimer le risque.*')

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown('### Caractéristiques du point')
                elev_in   = gr.Slider(0, 1000, value=50,    step=1,   label='Élévation (m)')
                slope_in  = gr.Slider(0, 15,   value=2,     step=0.1, label='Pente (degrés)')
                precip_in = gr.Slider(40, 250,  value=120,   step=1,   label='Précipitations (mm)')
                stream_in = gr.Slider(0, 15000, value=500,   step=50,  label='Distance cours d\'eau (m)')
                sea_in    = gr.Slider(0, 100000,value=5000,  step=100, label='Distance à la mer (m)')
                road_in   = gr.Slider(0, 50000, value=1000,  step=100, label='Distance aux routes (m)')
                cities_in = gr.Slider(0, 100000,value=10000, step=100, label='Distance aux villes (m)')
                lc_in     = gr.Dropdown([30, 40, 60, 80, 90], value=40,
                                        label='Occupation du sol (lc_code: 30=urbain, 40=agricole, 60=forêt, 80=humide, 90=eau)')
                btn = gr.Button('🔍 Prédire le risque d\'inondation', variant='primary', size='lg')

            with gr.Column(scale=2):
                gr.Markdown('### Résultats')
                with gr.Tabs():
                    with gr.Tab('Prédictions des modèles'):
                        pred_out = gr.Markdown()
                    with gr.Tab('Métriques d\'évaluation'):
                        met_out  = gr.Markdown()

        btn.click(
            fn=predict_and_metrics,
            inputs=[elev_in, slope_in, precip_in, stream_in,
                    sea_in, road_in, cities_in, lc_in],
            outputs=[pred_out, met_out]
        )

    print('✔ Interface Gradio prête.')
    demo.launch(share=True, debug=False)
else:
    print('⚠ Gradio non disponible ou modèle non chargé.')


**Interprétation :** L'interface démontre la valeur opérationnelle du projet. Un utilisateur non technique peut saisir les caractéristiques d'un point géographique et obtenir instantanément la prédiction de chacun des quatre modèles, accompagnée d'une décision par vote. L'onglet « Métriques » contextualise la fiabilité des prédictions avec les scores obtenus sur le jeu de test. La fonction `apply_feature_engineering()` — définie à l'Étape 1.3 et réutilisée ici — garantit que les transformations appliquées en prédiction sont identiques à celles de l'entraînement.


## Étape 7 — Discussion des Résultats et Conclusion

### Pourquoi un modèle surpasse les autres ?

Le problème de classification des inondations est fondamentalement **non-linéaire** : la relation entre les variables géographiques et le risque ne se réduit pas à une combinaison linéaire. XGBoost et Random Forest capturent ces non-linéarités naturellement grâce à leurs arbres de décision, tandis que la Régression Logistique est limitée à des frontières linéaires. Le mécanisme de *boosting* séquentiel de XGBoost — chaque arbre corrigeant les erreurs du précédent — lui permet de se concentrer sur les zones à risque ambigu, ce qui lui confère généralement un avantage sur les données tabulaires.

Le SVM, bien que performant en dimension élevée, présente un coût computationnel quadratique et une forte sensibilité aux hyperparamètres `C` et `gamma`. Sur des données géographiques avec outliers réels, la marge de tolérance aux erreurs doit être soigneusement calibrée par RandomizedSearchCV.

### Impact de la normalisation

La Régression Logistique minimise la log-vraisemblance via le produit scalaire w·x. Sans normalisation, une variable à grande amplitude (ex. : distance en mètres) domine artificiellement les coefficients. StandardScaler ramène toutes les variables à la même échelle (μ=0, σ=1), permettant une descente de gradient uniforme. MinMaxScaler, à l'inverse, est fortement biaisé par les outliers géographiques qui compriment la majorité des valeurs dans un intervalle très étroit. Les arbres de décision (RF, XGBoost) utilisent des seuils de comparaison ordinaux et ne sont donc pas sensibles à l'échelle absolue des variables — ce que le graphique de la section 3.4 confirme empiriquement.

### Conclusion générale

Ce projet présente un pipeline de Machine Learning complet pour la classification du risque d'inondation en Tunisie du Nord. Les résultats montrent que les méthodes ensemblistes (Random Forest, XGBoost) surpassent les approches linéaires (LR, SVM) sur ce type de données géographiques, grâce à leur capacité à modéliser des interactions complexes entre les variables. Le feature engineering a enrichi la représentation des données en capturant des relations physiques pertinentes (proximité à l'eau, saturation topographique). L'interface Gradio illustre la faisabilité d'un déploiement opérationnel accessible aux non-spécialistes.

Ce travail pourrait être étendu en intégrant des données temporelles (séries de précipitations) ou des images satellitaires pour améliorer la prédiction en contexte d'alerte précoce.

### Piste d'amélioration méthodologique : validation spatiale

Le dataset étant intrinsèquement géographique (chaque ligne correspond à un point du territoire), un `train_test_split` aléatoire classique peut donner une estimation de l'AUC trop optimiste : des points géographiquement proches (donc très similaires en altitude, pente, précipitations, occupation du sol) peuvent se retrouver à la fois dans le train et dans le test, ce qui facilite artificiellement la tâche du modèle. Une **validation croisée spatiale** (par exemple un découpage en blocs géographiques, ou en laissant de côté des zones entières à chaque fold) donnerait une mesure plus réaliste de la capacité du modèle à généraliser à des zones jamais vues. C'est l'étape recommandée avant d'envisager un déploiement MLOps complet (MLflow → FastAPI → Docker → CI/CD).
